In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
# Calculate aggregate MASE for each model variant.

data_direc = '../data/TSB-AD-U'
file_names = pd.read_csv('../data/File_List/TSB-AD-U-Eva.csv')['file_name'].to_numpy()

models = [('bolt_small', 'Chronos-Bolt-small'), ('bolt_base', 'Chronos-Bolt-base'), ('TimeMoE-50M', 'Time-MoE-base'), ('TimeMoE-200M', 'Time-MoE-large'), ('timesfm-1.0-200m', 'Timesfm-1.0'), ('timesfm-2.0-500m', 'Timesfm-2.0')]
wins = [64, 256, 512]

labels = []
results = []

for model, label in models:
    for win in wins:

        variation = f'{model}_{win}'
        labels.append(f'{label}-{win}')
        print(variation)

        score_direc = f'../results/intermediate/{variation}/scores'

        mases = []

        for file_name in file_names:

            df = pd.read_csv(f'{data_direc}/{file_name}').dropna()

            # For a fair comparison across different context window lengths, evaluation begins at the 513th data point.
            scores = np.load(f'{score_direc}/{os.path.splitext(file_name)[0]}.npy')[512:] # Anomaly scores are just the squared errors.

            # The normalization factor for MASE is the mean absolute difference of the data, 
            # which is equivalent to the mean absolute error of a naive forecasting method that always predicts the previous value, i.e., the Last-1 baseline.
            abs_diff = df['Data'].diff().abs().to_numpy()[512:]
            norm = abs_diff.mean()

            mase = np.sqrt(scores).mean() / norm # `np.sqrt(scores).mean()` gives the mean absolute error of the model's predictions, and dividing by `norm` gives the MASE.
            mases.append(mase)

        # Aggregate MASE is calculated as the geometric mean of the MASE values across all test files.
        mases = np.array(mases)
        result = np.exp(np.mean(np.log(mases)))
        print(result)

        results.append(result)

In [ ]:
# Calculate aggregate MASE for each baseline.

baselines = ['Last-1', 'Last-3', 'Centered-5']

for baseline in baselines:

    print(baseline)
    mases = []

    for file_name in file_names:

        df = pd.read_csv(f'{data_direc}/{file_name}').dropna()

        # Calculate the absolute errors of the baseline predictions. 
        # The errors are scaled by a factor of `window_size / (window_size - 1)` to account for the bias introduced by using a moving average that includes the current value itself.
        # The implementation may be a bit involved, but the aggregate MASE of `Last-1` being 1.0 serves as a sanity check for the correctness of the implementation.
        if baseline == 'Last-1':
            errors = np.abs((df['Data'].to_numpy() - df['Data'].rolling(window = 2, center = False).mean().bfill().ffill().to_numpy())) * 2 / (2 - 1)
        elif baseline == 'Last-3':
            errors = np.abs((df['Data'].to_numpy() - df['Data'].rolling(window = 4, center = False).mean().bfill().ffill().to_numpy())) * 4 / (4 - 1)
        else: # baseline == 'Centered-5'
            errors = np.abs((df['Data'].to_numpy() - df['Data'].rolling(window = 5, center = True).mean().bfill().ffill().to_numpy())) * 5 / (5 - 1)
        
        # For a fair comparison across different context window lengths, evaluation begins at the 513th data point.
        errors = errors[512:]

        # The normalization factor for MASE is the mean absolute difference of the data, 
        # which is equivalent to the mean absolute error of a naive forecasting method that always predicts the previous value, i.e., the Last-1 baseline.
        abs_diff = df['Data'].diff().abs().to_numpy()[512:]
        norm = abs_diff.mean()

        mase = errors.mean() / norm
        mases.append(mase)

    # Aggregate MASE is calculated as the geometric mean of the MASE values across all test files.
    mases = np.array(mases)
    result = np.exp(np.mean(np.log(mases)))
    print(result)

    labels.append(baseline)
    results.append(result)

In [ ]:
# Plot

plt.figure(figsize=(9, 7))
plt.rcParams.update({'font.size': 15})
bars = plt.barh(labels[::-1], results[::-1])

# Add text labels next to the bars
for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.01,             # x position (slightly beyond bar)
             bar.get_y() + bar.get_height()/2,  # y position (centered)
             f'{width:.2f}',          # formatted value
             va='center')

plt.xlabel('Aggregate Mean Absolute Scaled Error (MASE)')
plt.xlim(0, 1.5)
plt.show()